# Figure 10: Gemini 3, temperature in {0.5, 1.0, 1.5}

This notebook requires a run of the accessible-protocol (on gemini-3-flash-preview) with T in {0.5,1.5}

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.plots.style import apply_paper_style, model_label, model_color, ordered_models
apply_paper_style()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_accessible_professions, load_accessible_professions_temperatures, cumulative_unique_by_sample, try_load

main = try_load(load_accessible_professions, models=["gemini-3-flash-preview"], label="accessible (gemini-3-flash-preview, T=1.0)")
if main is not None:
    main = main.assign(temperature=1.0)
    if main.empty:
        print("Accessible data loaded but has no gemini-3-flash-preview rows.")
        main = None

temps = try_load(load_accessible_professions_temperatures, label="temperature ablation (T=0.5/1.5)")

parts = [df for df in (main, temps) if df is not None]
combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["temperature", "prompt_key", "query", "model_version"])
if combined.empty:
    print("No temperature-ablation data at all (neither T=1.0 nor T=0.5/1.5): Fig 10 will be blank.")


In [ ]:
from src.plots.style import FORMATS_ALL

if combined.empty:
    print("Skipping Fig 10: no data loaded.")
else:
    fig, axes = plt.subplots(1, 6, figsize=(18, 3.2), sharey=True)
    prompt_keys = [f"{fmt}_chemist" for fmt in FORMATS_ALL]  # matches Table 4's Gemini-3/chemist convention
    colors = {0.5: "#72B7B2", 1.0: "#4C78A8", 1.5: "#b95d1e"}
    empty_panels = []
    any_lines = False
    for ax, pk in zip(axes, prompt_keys):
        panel = combined[combined["prompt_key"] == pk]
        if panel.empty:
            empty_panels.append(pk)
            ax.text(0.5, 0.5, "no data", ha="center", va="center", fontsize=8, color="gray", transform=ax.transAxes)
        else:
            cum = cumulative_unique_by_sample(panel, group_cols=["temperature", "query"], order_col="sample_id")
            for t in sorted(colors):
                tsub = cum[cum["temperature"] == t].sort_values("x")
                if tsub.empty:
                    continue
                ax.plot(tsub["x"], tsub["cum_unique"], color=colors[t], label=f"T={t}", linewidth=1.4)
                any_lines = True
        ax.set_title(pk, fontsize=9)

    if empty_panels:
        print(f"No data for prompt(s): {empty_panels}")
    axes[0].set_ylabel("Cumulative unique valid names")
    if any_lines:
        axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
